In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files (they are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Convert timestamp to timezone-aware datetime (UTC->Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        # Some timestamps may already be numeric seconds; convert robustly
        df['_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Helper to build compact summary for metric/trace/log files
def build_timeseries_summary(df, file_label, name_col):
    total_rows = int(df.shape[0])
    unique_cmdb = sorted(df['cmdb_id'].dropna().unique())[:50]
    unique_names = sorted(df[name_col].dropna().unique())[:50]
    # group by component and KPI/feature name
    gp = df.groupby(['cmdb_id', name_col], dropna=False)['value']
    agg = gp.agg(count='count', p95=lambda s: float(np.nanpercentile(s.dropna(), 95)) if s.dropna().shape[0] else np.nan,
                 min_value='min', max_value='max').reset_index()
    # earliest timestamp per group (use _dt)
    if '_dt' in df.columns:
        earliest = df.groupby(['cmdb_id', name_col], dropna=False)['_dt'].min().reset_index().rename(columns={'_dt':'earliest_ts'})
        agg = agg.merge(earliest, on=['cmdb_id', name_col], how='left')
    # sort by count desc and limit top 50 for display
    agg = agg.sort_values('count', ascending=False).reset_index(drop=True)
    top_agg = agg.head(50).copy()
    # round numeric columns for compactness
    for col in ('p95','min_value','max_value'):
        if col in top_agg.columns:
            top_agg[col] = top_agg[col].round(6)
    # summary single-row
    summary = pd.DataFrame([{
        'file': file_label,
        'total_rows': total_rows,
        'unique_cmdb_count': len(sorted(df['cmdb_id'].dropna().unique())),
        'unique_cmdb_sample': unique_cmdb,
        'unique_name_count': len(sorted(df[name_col].dropna().unique())),
        'unique_name_sample': unique_names
    }])
    return summary, top_agg

# Build summaries
metric_summary_df, metric_counts_df = build_timeseries_summary(metric_df, 'metric.csv', 'kpi_name')
trace_summary_df, trace_counts_df = build_timeseries_summary(trace_df, 'trace.csv', 'trace_name')
log_summary_df, log_counts_df = build_timeseries_summary(log_df, 'log.csv', 'log_name')

# error_logs.csv summary (no 'value' KPI series)
error_total_rows = int(error_df.shape[0])
error_unique_cmdb = sorted(error_df['cmdb_id'].dropna().unique())[:50]
# Use 'message' as the representative field for unique messages (limit 50)
if 'message' in error_df.columns:
    error_unique_messages = sorted(error_df['message'].dropna().unique())[:50]
else:
    error_unique_messages = []
error_earliest_ts = error_df['_dt'].min() if '_dt' in error_df.columns else pd.NaT

error_summary_df = pd.DataFrame([{
    'file': 'error_logs.csv',
    'total_rows': error_total_rows,
    'unique_cmdb_count': len(sorted(error_df['cmdb_id'].dropna().unique())),
    'unique_cmdb_sample': error_unique_cmdb,
    'unique_message_count': len(sorted(error_df['message'].dropna().unique())) if 'message' in error_df.columns else 0,
    'unique_message_sample': error_unique_messages,
    'earliest_timestamp': error_earliest_ts
}])

# Final displayed outputs (compact)
metric_summary_df, metric_counts_df, trace_summary_df, trace_counts_df, log_summary_df, log_counts_df, error_summary_df

```
Out[1]:
```
import pandas as pd

# Reuse previously computed summary DataFrames from the kernel:
# metric_summary_df, metric_counts_df, trace_summary_df, trace_counts_df, log_summary_df, log_counts_df, error_summary_df

# Helper to format top rows for a timeseries counts DataFrame
def format_top_rows(df, name_col, top_n=3):
    rows = []
    for _, r in df.head(top_n).iterrows():
        comp = r['cmdb_id']
        name = r[name_col]
        cnt = int(r['count'])
        p95 = float(r['p95']) if pd.notna(r['p95']) else None
        ts = r['earliest_ts']
        ts_str = ts.isoformat() if pd.notna(ts) and hasattr(ts, 'isoformat') else str(ts)
        rows.append(f"{comp} | {name} | count={cnt} | p95={p95} | earliest={ts_str}")
    return rows

# Build the plain-English summary
m = metric_summary_df.loc[0]
t = trace_summary_df.loc[0]
l = log_summary_df.loc[0]
e = error_summary_df.loc[0]

metric_top = format_top_rows(metric_counts_df, 'kpi_name', top_n=3)
trace_top = format_top_rows(trace_counts_df, 'trace_name', top_n=3)
log_top = format_top_rows(log_counts_df, 'log_name', top_n=3)

summary_lines = [
    "Summary of loaded telemetry files (UTC+8):",
    "",
    f"metric.csv: total_rows={int(m.total_rows)}, unique_components={int(m.unique_cmdb_count)}, unique_kpis={int(m.unique_name_count)}",
    f"  sample components: {m.unique_cmdb_sample[:8]}",
    f"  sample KPI names (first 8): {m.unique_name_sample[:8]}",
    "  Top 3 (component | kpi_name | count | p95 | earliest):",
] + [f"    - {r}" for r in metric_top] + [
    "",
    f"trace.csv: total_rows={int(t.total_rows)}, unique_components={int(t.unique_cmdb_count)}, unique_trace_names={int(t.unique_name_count)}",
    f"  sample components: {t.unique_cmdb_sample[:8]}",
    f"  sample trace names (first 8): {t.unique_name_sample[:8]}",
    "  Top 3 (component | trace_name | count | p95 | earliest):",
] + [f"    - {r}" for r in trace_top] + [
    "",
    f"log.csv: total_rows={int(l.total_rows)}, unique_components={int(l.unique_cmdb_count)}, unique_log_names={int(l.unique_name_count)}",
    f"  sample components: {l.unique_cmdb_sample[:8]}",
    f"  sample log names (first 8): {l.unique_name_sample[:8]}",
    "  Top 3 (component | log_name | count | p95 | earliest):",
] + [f"    - {r}" for r in log_top] + [
    "",
    f"error_logs.csv: total_rows={int(e.total_rows)}, unique_components={int(e.unique_cmdb_count)}, unique_messages={int(e.unique_message_count)}",
    f"  sample components: {e.unique_cmdb_sample}",
    f"  earliest_error_timestamp={e.earliest_timestamp}",
    "",
    "Notes: Counts and global P95 values were computed across the entire file for each (component, KPI/trace/log) pair. Display limited to top 3 pairs per file for brevity."
]

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows  unique_cmdb_count                                 unique_cmdb_sample  unique_name_count                                 unique_name_sample
0  metric.csv       28244                 29  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...                351  [Container-DOCKER_CONTAINER_2d16f5b2e830--bcou...,      cmdb_id                                           kpi_name  count        p95  min_value  max_value               earliest_ts
0   apache01                         OSLinux-CPU_CPU_CPUSysTime     31  18.444244  18.168600  18.565453 2021-03-07 07:30:00+08:00
1   apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31   0.000000   0.000000   0.000000 2021-03-07 07:30:00+08:00
2       IG01                        OSLinux-CPU_CPU_CPUidleutil     31  74.563000  73.764085  74.638300 2021-03-07 07:30:00+08:00
3    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31   0.000000   0.000000   0.000000 2021-03-07 07:30:00+08:00
4    Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31   0.400000   0.400000   0.400000 2021-03-07 07:30:00+08:00
..       ...                                                ...    ...        ...        ...        ...                       ...
45  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31   0.000000   0.000000   0.000000 2021-03-07 07:30:00+08:00
46  Tomcat02        Tomcat-Sessions_7441--_SESSIONActiveCounter     31   0.000000   0.000000   0.000000 2021-03-07 07:30:00+08:00
47      IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31   0.000000   0.000000   0.000000 2021-03-07 07:30:00+08:00
48  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31   0.000000   0.000000   0.000000 2021-03-07 07:30:00+08:00
49  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31   0.000000   0.000000   0.000000 2021-03-07 07:30:00+08:00

[50 rows x 7 columns],         file  total_rows  unique_cmdb_count                                 unique_cmdb_sample  unique_name_count                                 unique_name_sample
0  trace.csv        6840                 12  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...                 72  [trace.from_IG01.duration_mean, trace.from_IG0...,    cmdb_id                         trace_name  count         p95   min_value   max_value               earliest_ts
0     IG01           trace.self.duration_mean     30   12.035910    0.050186   16.852853 2021-03-07 07:30:00+08:00
1     IG01            trace.self.duration_p95     30   45.199600    0.125000   45.255050 2021-03-07 07:30:00+08:00
2     IG01               trace.self.row_count     30  328.600000  102.000000  344.000000 2021-03-07 07:30:00+08:00
3     IG01    trace.to_Tomcat01.duration_mean     30    0.126800    0.028571    0.151586 2021-03-07 07:30:00+08:00
4     IG01     trace.to_Tomcat01.duration_p95     30    0.690120    0.059700    0.926800 2021-03-07 07:30:00+08:00
..     ...                                ...    ...         ...         ...         ...                       ...
45    MG01   trace.from_dockerB1.duration_p95     30    0.172035    0.008000    0.756400 2021-03-07 07:30:00+08:00
46    MG01  trace.from_dockerB1.duration_mean     30    0.057638    0.006042    0.145222 2021-03-07 07:30:00+08:00
47    MG01   trace.from_dockerB2.duration_p95     30    0.036552    0.005850    0.128750 2021-03-07 07:30:00+08:00
48    MG01  trace.from_dockerB2.duration_mean     30    0.033209    0.005250    0.040433 2021-03-07 07:30:00+08:00
49    MG01      trace.from_dockerB1.row_count     30   59.300000    9.000000   65.000000 2021-03-07 07:30:00+08:00

[50 rows x 7 columns],       file  total_rows  unique_cmdb_count                                 unique_cmdb_sample  unique_name_count                                 unique_name_sample
0  log.csv         407                  7  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...                  3  [log.error_count, log.gc_pause_max, log.row_co...,      cmdb_id          log_name  count       p95  min_value  max_value               earliest_ts
0   apache02     log.row_count     31  160.5000       1.00     164.00 2021-03-07 07:30:00+08:00
1   Tomcat03   log.error_count     31    0.0000       0.00       0.00 2021-03-07 07:30:00+08:00
2   apache01   log.error_count     31    0.0000       0.00       5.00 2021-03-07 07:30:00+08:00
3   Tomcat04     log.row_count     31   86.5000       1.00      88.00 2021-03-07 07:30:00+08:00
4   apache01     log.row_count     31  171.0000       3.00     178.00 2021-03-07 07:30:00+08:00
5   apache02   log.error_count     31    0.0000       0.00       3.00 2021-03-07 07:30:00+08:00
6   Tomcat04   log.error_count     31    0.0000       0.00       0.00 2021-03-07 07:30:00+08:00
7   Tomcat03     log.row_count     31   83.0000       5.00      85.00 2021-03-07 07:30:00+08:00
8   Tomcat01     log.row_count     30   90.3000      22.00     104.00 2021-03-07 07:30:00+08:00
9   Tomcat02   log.error_count     30    0.0000       0.00       0.00 2021-03-07 07:30:00+08:00
10  Tomcat01   log.error_count     30    0.0000       0.00       0.00 2021-03-07 07:30:00+08:00
11  Tomcat02     log.row_count     30   86.2000      30.00      90.00 2021-03-07 07:30:00+08:00
12  Tomcat02  log.gc_pause_max     10    0.0900       0.03       0.09 2021-03-07 07:32:00+08:00
13   Mysql01   log.error_count      8    0.0000       0.00       0.00 2021-03-07 07:30:00+08:00
14   Mysql01     log.row_count      8    1.0000       1.00       1.00 2021-03-07 07:30:00+08:00
15  Tomcat04  log.gc_pause_max      5    0.0980       0.06       0.10 2021-03-07 07:33:00+08:00
16  Tomcat03  log.gc_pause_max      4    0.0800       0.05       0.08 2021-03-07 07:35:00+08:00
17  Tomcat01  log.gc_pause_max      4    0.0685       0.02       0.07 2021-03-07 07:36:00+08:00,              file  total_rows  unique_cmdb_count    unique_cmdb_sample  unique_message_count                              unique_message_sample        earliest_timestamp
0  error_logs.csv           8                  2  [apache01, apache02]                     8  [IPAddress "POST /UOCP/base/ServiceTest11.json... 2021-03-07 07:36:21+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables from kernel (metric_df, trace_df, log_df, error_df,
# metric_counts_df, trace_counts_df, log_counts_df) which were loaded earlier.

# Ensure timezone (UTC+8) and window boundaries
tz = pytz.timezone('Asia/Shanghai')
window_start = pd.Timestamp('2021-03-07 07:30:00').tz_localize(tz)
window_end = pd.Timestamp('2021-03-07 08:00:00').tz_localize(tz)
tiny_eps = 1e-9

# Windowed DataFrames (keep for later use)
metric_win = metric_df[(metric_df['_dt'] >= window_start) & (metric_df['_dt'] <= window_end)].copy()
trace_win = trace_df[(trace_df['_dt'] >= window_start) & (trace_df['_dt'] <= window_end)].copy()
log_win = log_df[(log_df['_dt'] >= window_start) & (log_df['_dt'] <= window_end)].copy()
error_win = error_df[(error_df['_dt'] >= window_start) & (error_df['_dt'] <= window_end)].copy()

# Helper to compute anomalies using precomputed global p95 per (component, name)
def compute_anomalies(window_df, global_p95_df, name_col, value_col='value'):
    # Merge to bring p95 into each row; only keep pairs with stored global p95
    merged = window_df.merge(
        global_p95_df[['cmdb_id', name_col, 'p95']],
        on=['cmdb_id', name_col],
        how='inner',
        validate='m:1'
    ).copy()
    if merged.empty:
        # return empty aggregated frame
        agg_empty = pd.DataFrame(columns=['cmdb_id', name_col, 'total_points_in_window',
                                          'anomaly_count', 'earliest_anomaly_timestamp',
                                          'p95', 'max_value_in_window', 'severity'])
        return merged, agg_empty

    # Determine anomaly per row
    merged['is_anomaly'] = np.where(merged['p95'] == 0, merged[value_col] > 0, merged[value_col] > merged['p95'])

    # Group-level aggregates
    grp = merged.groupby(['cmdb_id', name_col], dropna=False)

    total_points = grp[value_col].count().rename('total_points_in_window')
    anomaly_counts = grp['is_anomaly'].sum().rename('anomaly_count')
    max_values = grp[value_col].max().rename('max_value_in_window')
    p95_first = grp['p95'].first().rename('p95')

    # earliest anomaly timestamp: compute from rows that are anomalies
    anomalies_only = merged[merged['is_anomaly']].copy()
    if not anomalies_only.empty:
        earliest_anom_ts = anomalies_only.groupby(['cmdb_id', name_col])['_dt'].min().rename('earliest_anomaly_timestamp')
        agg = pd.concat([total_points, anomaly_counts, max_values, p95_first, earliest_anom_ts], axis=1).reset_index()
    else:
        # No anomalies at all
        agg = pd.concat([total_points, anomaly_counts, max_values, p95_first], axis=1).reset_index()
        agg['earliest_anomaly_timestamp'] = pd.NaT

    # Keep only rows with anomaly_count > 0
    agg = agg[agg['anomaly_count'] > 0].copy()
    if agg.empty:
        agg = agg.reset_index(drop=True)
        agg = agg[['cmdb_id', name_col, 'total_points_in_window', 'anomaly_count',
                   'earliest_anomaly_timestamp', 'p95', 'max_value_in_window']]

    # Severity calculation
    agg['severity'] = agg['max_value_in_window'] / agg['p95'].replace(0, tiny_eps)

    # Rounding for compactness
    agg['p95'] = agg['p95'].astype(float).round(6)
    agg['max_value_in_window'] = agg['max_value_in_window'].astype(float).round(6)
    agg['severity'] = agg['severity'].astype(float).round(6)

    # Sort by anomaly_count desc then severity desc
    agg = agg.sort_values(['anomaly_count', 'severity'], ascending=[False, False]).reset_index(drop=True)
    return merged, agg

# Compute anomalies for metric, trace, log
metric_merged_win, metric_anomalies = compute_anomalies(metric_win, metric_counts_df, 'kpi_name')
trace_merged_win, trace_anomalies = compute_anomalies(trace_win, trace_counts_df, 'trace_name')
log_merged_win, log_anomalies = compute_anomalies(log_win, log_counts_df, 'log_name')

# Prepare display tables (top 20 rows)
metric_anomalies_display = metric_anomalies[['cmdb_id', 'kpi_name', 'total_points_in_window',
                                             'anomaly_count', 'earliest_anomaly_timestamp',
                                             'p95', 'max_value_in_window', 'severity']].head(20)

trace_anomalies_display = trace_anomalies[['cmdb_id', 'trace_name', 'total_points_in_window',
                                           'anomaly_count', 'earliest_anomaly_timestamp',
                                           'p95', 'max_value_in_window', 'severity']].head(20)

log_anomalies_display = log_anomalies[['cmdb_id', 'log_name', 'total_points_in_window',
                                       'anomaly_count', 'earliest_anomaly_timestamp',
                                       'p95', 'max_value_in_window', 'severity']].head(20)

# Error logs within window: show timestamp, cmdb_id, message, limit 50, and earliest timestamp within window
error_win_display = error_win[['timestamp', 'cmdb_id', 'message', '_dt']].sort_values('_dt').head(50)
earliest_error_in_window = error_win['_dt'].min() if not error_win.empty else pd.NaT

# Keep full filtered tables in memory for later steps:
# metric_win, trace_win, log_win, error_win,
# metric_merged_win, trace_merged_win, log_merged_win,
# metric_anomalies, trace_anomalies, log_anomalies

# Final compact outputs (display variables)
metric_anomalies_display, trace_anomalies_display, log_anomalies_display, error_win_display, earliest_error_in_window

```
Out[2]:
```
```python
# Summarize the anomaly findings in plain English using previously computed variables in the kernel.
# This cell produces a single summary string variable `summary` as the output.

# Reuse kernel variables:
# metric_anomalies, trace_anomalies, log_anomalies, metric_anomalies_display,
# trace_anomalies_display, log_anomalies_display, error_win, earliest_error_in_window

# Prepare counts
n_metric_pairs = int(metric_anomalies.shape[0]) if 'metric_anomalies' in globals() else 0
n_trace_pairs = int(trace_anomalies.shape[0]) if 'trace_anomalies' in globals() else 0
n_log_pairs = int(log_anomalies.shape[0]) if 'log_anomalies' in globals() else 0
n_error_rows = int(error_win.shape[0]) if 'error_win' in globals() else 0

# Helper to format top N rows for a DataFrame into compact lines
def top_lines(df, name_col, top_n=3):
    lines = []
    if df is None or df.shape[0] == 0:
        return ["  (none)"]
    for _, r in df.head(top_n).iterrows():
        comp = r['cmdb_id']
        name = r[name_col]
        cnt = int(r['anomaly_count'])
        p95 = float(r['p95'])
        maxv = float(r['max_value_in_window'])
        sev = float(r['severity'])
        ts = r['earliest_anomaly_timestamp']
        ts_str = ts.isoformat() if pd.notna(ts) and hasattr(ts, 'isoformat') else str(ts)
        lines.append(f"  - {comp} | {name} | anomalies={cnt} | p95={p95:.6g} | max={maxv:.6g} | sev={sev:.6g} | first={ts_str}")
    return lines

metric_top_lines = top_lines(metric_anomalies, 'kpi_name', top_n=5)
trace_top_lines = top_lines(trace_anomalies, 'trace_name', top_n=5)
log_top_lines = top_lines(log_anomalies, 'log_name', top_n=5)

# Earliest error timestamp in window
earliest_error_ts = earliest_error_in_window if 'earliest_error_in_window' in globals() else None
earliest_error_str = earliest_error_ts.isoformat() if pd.notna(earliest_error_ts) and hasattr(earliest_error_ts, 'isoformat') else str(earliest_error_ts)

summary_lines = [
    "Summary of anomalies and errors within 2021-03-07 07:30:00+08 to 2021-03-07 08:00:00+08:",
    "",
    f"1) metric.csv: {n_metric_pairs} (component,kpi) pairs had at least one anomaly in the window.",
    " Top examples:",
] + metric_top_lines + [
    "",
    f"2) trace.csv: {n_trace_pairs} (component,trace_name) pairs had at least one anomaly in the window.",
    " Top examples:",
] + trace_top_lines + [
    "",
    f"3) log.csv: {n_log_pairs} (component,log_name) pairs had at least one anomaly in the window.",
    " Top examples:",
] + log_top_lines + [
    "",
    f"4) error_logs.csv: {n_error_rows} error rows found in the window (showing up to 50).",
    f" Earliest error timestamp in window: {earliest_error_str}",
    "",
    "Notes:",
    " - An anomaly is defined as value > global P95; if P95 == 0 then value > 0 is treated as anomaly.",
    " - Severity = max_value_in_window / max(p95, 1e-9).",
    " - Top examples list component, metric name, anomaly count, p95, max value, severity, and first anomaly time.",
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp           p95  max_value_in_window  severity
0       IG01                             OSLinux-CPU_CPU_CPUWio                      31              2  2021-03-07 07:40:00+08:00  2.640000e-01         5.737810e-01  2.173413
1    Redis02                             OSLinux-CPU_CPU_CPUWio                      31              2  2021-03-07 07:38:00+08:00  3.535000e-02         7.070000e-02  2.000000
2   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31              2  2021-03-07 07:40:00+08:00  5.000000e-01         1.000000e+00  2.000000
3    Mysql01                            OSLinux-CPU_CPU_CPULoad                      31              2  2021-03-07 07:38:00+08:00  1.400000e-01         1.600000e-01  1.142857
4    Redis02                        OSLinux-CPU_CPU_CPUUserTime                      31              2  2021-03-07 07:40:00+08:00  2.309420e-01         2.579000e-01  1.116731
5    Mysql01                             OSLinux-CPU_CPU_CPUWio                      31              2  2021-03-07 07:42:00+08:00  4.403710e-01         4.903000e-01  1.113379
6       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed                      31              2  2021-03-07 07:50:00+08:00  7.802229e+08         8.423134e+08  1.079580
7    Redis02                         OSLinux-CPU_CPU_CPUSysTime                      31              2  2021-03-07 07:40:00+08:00  4.909350e-01         5.242000e-01  1.067758
8       IG01                         OSLinux-CPU_CPU_CPUSysTime                      31              2  2021-03-07 07:36:00+08:00  3.928820e-01         4.115000e-01  1.047388
9    Mysql01                         OSLinux-CPU_CPU_CPUSysTime                      31              2  2021-03-07 07:33:00+08:00  4.092000e-01         4.279360e-01  1.045787
10      IG01                            OSLinux-CPU_CPU_CPULoad                      31              2  2021-03-07 07:34:00+08:00  1.025000e+00         1.070000e+00  1.043902
11      MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed                      31              2  2021-03-07 07:30:00+08:00  1.165741e+09         1.209754e+09  1.037756
12   Mysql01                        OSLinux-CPU_CPU_CPUUserTime                      31              2  2021-03-07 07:43:00+08:00  2.409640e-01         2.452000e-01  1.017579
13  apache01                         OSLinux-CPU_CPU_CPUSysTime                      31              2  2021-03-07 07:36:00+08:00  1.844424e+01         1.856545e+01  1.006572
14      IG01                        OSLinux-CPU_CPU_CPUUserTime                      31              2  2021-03-07 07:43:00+08:00  2.518240e+01         2.521515e+01  1.001301
15      IG01                        OSLinux-CPU_CPU_CPUidleutil                      31              2  2021-03-07 07:32:00+08:00  7.456300e+01         7.463830e+01  1.001010
16   Mysql01                        OSLinux-CPU_CPU_CPUidleutil                      31              2  2021-03-07 07:35:00+08:00  9.938310e+01         9.946400e+01  1.000814
17   Redis02                        OSLinux-CPU_CPU_CPUidleutil                      31              2  2021-03-07 07:37:00+08:00  9.956339e+01         9.958410e+01  1.000208
18  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...                      31              2  2021-03-07 07:59:00+08:00  5.432380e+05         5.433450e+05  1.000197
19  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...                      31              2  2021-03-07 07:59:00+08:00  3.299632e+08         3.299783e+08  1.000046,    cmdb_id                         trace_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp        p95  max_value_in_window  severity
0     MG01   trace.from_dockerB1.duration_p95                      30              2  2021-03-07 07:31:00+08:00   0.172035             0.756400  4.396780
1     MG01   trace.from_dockerB2.duration_p95                      30              2  2021-03-07 07:33:00+08:00   0.036552             0.128750  3.522379
2     MG01  trace.from_dockerB1.duration_mean                      30              2  2021-03-07 07:31:00+08:00   0.057638             0.145222  2.519553
3     IG02            trace.self.duration_p95                      30              2  2021-03-07 07:33:00+08:00  45.200480           105.014000  2.323294
4     IG01     trace.to_Tomcat04.duration_p95                      30              2  2021-03-07 07:39:00+08:00   0.522955             0.834000  1.594783
5     MG01   trace.from_dockerA1.duration_p95                      30              2  2021-03-07 07:45:00+08:00   0.019877             0.030700  1.544499
6     MG01  trace.from_Tomcat01.duration_mean                      30              2  2021-03-07 07:31:00+08:00   0.121170             0.185636  1.532029
7     MG01   trace.from_Tomcat01.duration_p95                      30              2  2021-03-07 07:44:00+08:00   0.815305             1.148000  1.408062
8     IG01           trace.self.duration_mean                      30              2  2021-03-07 07:33:00+08:00  12.035910            16.852853  1.400214
9     IG01     trace.to_Tomcat01.duration_p95                      30              2  2021-03-07 07:31:00+08:00   0.690120             0.926800  1.342955
10    IG01    trace.to_Tomcat02.duration_mean                      30              2  2021-03-07 07:42:00+08:00   0.188276             0.252021  1.338572
11    MG01      trace.from_Tomcat01.row_count                      30              2  2021-03-07 07:43:00+08:00  47.550000            61.000000  1.282860
12    MG01  trace.from_Tomcat02.duration_mean                      30              2  2021-03-07 07:42:00+08:00   0.167448             0.213148  1.272921
13    IG02     trace.to_Tomcat01.duration_p95                      30              2  2021-03-07 07:44:00+08:00   1.042250             1.305300  1.252387
14    IG01    trace.to_Tomcat03.duration_mean                      30              2  2021-03-07 07:43:00+08:00   0.176748             0.217655  1.231443
15    MG01  trace.from_dockerB2.duration_mean                      30              2  2021-03-07 07:43:00+08:00   0.033209             0.040433  1.217531
16    MG01   trace.from_Tomcat03.duration_p95                      30              2  2021-03-07 07:41:00+08:00   0.917305             1.116300  1.216934
17    IG02        trace.to_Tomcat02.row_count                      30              2  2021-03-07 07:43:00+08:00  42.100000            51.000000  1.211401
18    IG01     trace.to_Tomcat02.duration_p95                      30              2  2021-03-07 07:42:00+08:00   1.028577             1.233800  1.199521
19    IG01    trace.to_Tomcat01.duration_mean                      30              2  2021-03-07 07:31:00+08:00   0.126800             0.151586  1.195473,     cmdb_id          log_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp       p95  max_value_in_window      severity
0  Tomcat01     log.row_count                      30              2  2021-03-07 07:49:00+08:00   90.3000               104.00  1.151717e+00
1  Tomcat02     log.row_count                      30              2  2021-03-07 07:42:00+08:00   86.2000                90.00  1.044084e+00
2  apache01     log.row_count                      31              2  2021-03-07 07:42:00+08:00  171.0000               178.00  1.040936e+00
3  Tomcat03     log.row_count                      31              2  2021-03-07 07:42:00+08:00   83.0000                85.00  1.024096e+00
4  apache02     log.row_count                      31              2  2021-03-07 07:42:00+08:00  160.5000               164.00  1.021807e+00
5  Tomcat04     log.row_count                      31              2  2021-03-07 07:42:00+08:00   86.5000                88.00  1.017341e+00
6  apache01   log.error_count                      31              1  2021-03-07 07:36:00+08:00    0.0000                 5.00  5.000000e+09
7  apache02   log.error_count                      31              1  2021-03-07 07:36:00+08:00    0.0000                 3.00  3.000000e+09
8  Tomcat01  log.gc_pause_max                       4              1  2021-03-07 07:49:00+08:00    0.0685                 0.07  1.021898e+00
9  Tomcat04  log.gc_pause_max                       5              1  2021-03-07 07:41:00+08:00    0.0980                 0.10  1.020408e+00,     timestamp   cmdb_id                                            message                       _dt
0  1615073781  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H... 2021-03-07 07:36:21+08:00
1  1615073781  apache01  IPAddress "POST /UOCP/person/ServiceTest10.jso... 2021-03-07 07:36:21+08:00
2  1615073781  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H... 2021-03-07 07:36:21+08:00
3  1615073782  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H... 2021-03-07 07:36:22+08:00
4  1615073783  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json... 2021-03-07 07:36:23+08:00
5  1615073784  apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ... 2021-03-07 07:36:24+08:00
6  1615073785  apache02  IPAddress "POST /UOCP/base/ServiceTest2.json H... 2021-03-07 07:36:25+08:00
7  1615073785  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso... 2021-03-07 07:36:25+08:00, Timestamp('2021-03-07 07:36:21+0800', tz='Asia/Shanghai'))```
```

In [ ]:
# In[3]:

# Filter trace anomalies within the given window and produce aggregate and raw outputs.
# Reuse kernel variables: trace_df, trace_win, trace_counts_df (contains global p95 per pair)
import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
tiny_eps = 1e-9

# Ensure trace_counts_df is available (contains global p95 per (cmdb_id, trace_name))
# Merge p95 into all rows in the window for trace series (only pairs that have stored p95 will be kept)
merged_trace_win = trace_win.merge(
    trace_counts_df[['cmdb_id', 'trace_name', 'p95']],
    on=['cmdb_id', 'trace_name'],
    how='inner',
    validate='m:1'
).copy()

# Determine anomaly rows: value > p95 (or >0 if p95 == 0)
if not merged_trace_win.empty:
    merged_trace_win['is_anomaly'] = np.where(merged_trace_win['p95'] == 0,
                                             merged_trace_win['value'] > 0,
                                             merged_trace_win['value'] > merged_trace_win['p95'])
else:
    merged_trace_win['is_anomaly'] = pd.Series(dtype=bool)

# Store full filtered anomaly rows in kernel variable for later steps
trace_anomaly_rows = merged_trace_win[merged_trace_win['is_anomaly']].copy()

# Aggregated table per (cmdb_id, trace_name)
if not merged_trace_win.empty:
    # total points in window per pair (only pairs with stored p95 are included due to merge)
    total_points = merged_trace_win.groupby(['cmdb_id', 'trace_name'])['value'].count().rename('total_points_in_window')
    # anomaly count per pair (from is_anomaly)
    anomaly_count = merged_trace_win.groupby(['cmdb_id', 'trace_name'])['is_anomaly'].sum().rename('anomaly_count')
    # p95 per pair (first)
    p95_series = merged_trace_win.groupby(['cmdb_id', 'trace_name'])['p95'].first().rename('p95')
    # max value in window per pair
    max_value = merged_trace_win.groupby(['cmdb_id', 'trace_name'])['value'].max().rename('max_value_in_window')
    agg = pd.concat([total_points, anomaly_count, p95_series, max_value], axis=1).reset_index()
    # earliest anomaly timestamp per pair (from anomaly rows)
    if not trace_anomaly_rows.empty:
        earliest_anom = trace_anomaly_rows.groupby(['cmdb_id', 'trace_name'])['_dt'].min().rename('earliest_anomaly_timestamp').reset_index()
        agg = agg.merge(earliest_anom, on=['cmdb_id', 'trace_name'], how='left')
    else:
        agg['earliest_anomaly_timestamp'] = pd.NaT

    # Keep only pairs with at least one anomaly
    agg = agg[agg['anomaly_count'] > 0].copy()

    # Compute severity = max_value_in_window / max(p95, tiny_eps)
    agg['severity'] = agg['max_value_in_window'] / agg['p95'].replace(0, tiny_eps)

    # Rounding for compactness
    agg['p95'] = agg['p95'].astype(float).round(6)
    agg['max_value_in_window'] = agg['max_value_in_window'].astype(float).round(6)
    agg['severity'] = agg['severity'].astype(float).round(6)

    # Order by anomaly_count desc then severity desc, limit top 20
    trace_agg_display = agg.sort_values(['anomaly_count', 'severity'], ascending=[False, False]).head(20).reset_index(drop=True)
else:
    trace_agg_display = pd.DataFrame(columns=['cmdb_id','trace_name','total_points_in_window','anomaly_count',
                                              'earliest_anomaly_timestamp','p95','max_value_in_window','severity'])

# Raw anomaly rows sorted by timestamp ascending, limited to top 50, show requested columns
if not trace_anomaly_rows.empty:
    trace_anomaly_rows_sorted = trace_anomaly_rows.sort_values('_dt').reset_index(drop=True)
    trace_anomaly_rows_display = trace_anomaly_rows_sorted[['timestamp', 'cmdb_id', 'trace_name', 'value', 'p95']].head(50)
else:
    trace_anomaly_rows_display = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95'])

# Keep full filtered anomalies in variable trace_anomaly_rows (already set) for later steps.
# Final displayed outputs (compact)
trace_agg_display, trace_anomaly_rows_display

```
Out[3]:
```
# Summarize trace anomalies within 2021-03-07 07:30:00+08 to 2021-03-07 08:00:00+08.
# Uses kernel variables: trace_anomaly_rows (full filtered anomalies), trace_agg_display (top aggregated rows).
import pandas as pd

# Safely access variables from kernel
try:
    full_anom = trace_anomaly_rows
except NameError:
    full_anom = pd.DataFrame()

try:
    agg_top = trace_agg_display
except NameError:
    agg_top = pd.DataFrame()

# Compute basic counts and earliest timestamp
n_anomaly_rows = int(full_anom.shape[0])
n_pairs = int(full_anom[['cmdb_id','trace_name']].drop_duplicates().shape[0]) if not full_anom.empty else 0
earliest_ts = full_anom['_dt'].min() if not full_anom.empty else pd.NaT
earliest_str = earliest_ts.isoformat() if pd.notna(earliest_ts) else "None"

# Prepare top aggregated lines (use agg_top which is already limited to top 20)
top_lines = []
if not agg_top.empty:
    for _, r in agg_top.head(5).iterrows():
        comp = r['cmdb_id']
        name = r['trace_name']
        cnt = int(r['anomaly_count'])
        p95 = float(r['p95'])
        maxv = float(r['max_value_in_window'])
        sev = float(r['severity'])
        ts = r.get('earliest_anomaly_timestamp', None)
        ts_str = ts.isoformat() if pd.notna(ts) else "None"
        top_lines.append(f"{comp} | {name} | anomalies={cnt} | p95={p95:.6g} | max={maxv:.6g} | sev={sev:.6g} | first={ts_str}")
else:
    top_lines.append("  (no aggregated anomalies)")

summary_lines = [
    "Trace anomalies summary (window: 2021-03-07 07:30:00+08 to 2021-03-07 08:00:00+08):",
    f"- Total anomaly rows found: {n_anomaly_rows}",
    f"- Distinct (cmdb_id, trace_name) pairs with anomalies: {n_pairs}",
    f"- Earliest anomaly timestamp in window: {earliest_str}",
    "",
    "Top aggregated (component | trace_name | anomalies | p95 | max | severity | first):"
] + [f"  - {ln}" for ln in top_lines] + [
    "",
    "Notes:",
    " - Full filtered anomaly rows are stored in the kernel variable `trace_anomaly_rows`.",
    " - A sample of raw anomaly rows (up to 50) was produced earlier as `trace_anomaly_rows_display`.",
]

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                         trace_name  total_points_in_window  anomaly_count        p95  max_value_in_window earliest_anomaly_timestamp  severity
0     MG01   trace.from_dockerB1.duration_p95                      30              2   0.172035             0.756400  2021-03-07 07:31:00+08:00  4.396780
1     MG01   trace.from_dockerB2.duration_p95                      30              2   0.036552             0.128750  2021-03-07 07:33:00+08:00  3.522379
2     MG01  trace.from_dockerB1.duration_mean                      30              2   0.057638             0.145222  2021-03-07 07:31:00+08:00  2.519553
3     IG02            trace.self.duration_p95                      30              2  45.200480           105.014000  2021-03-07 07:33:00+08:00  2.323294
4     IG01     trace.to_Tomcat04.duration_p95                      30              2   0.522955             0.834000  2021-03-07 07:39:00+08:00  1.594783
5     MG01   trace.from_dockerA1.duration_p95                      30              2   0.019877             0.030700  2021-03-07 07:45:00+08:00  1.544499
6     MG01  trace.from_Tomcat01.duration_mean                      30              2   0.121170             0.185636  2021-03-07 07:31:00+08:00  1.532029
7     MG01   trace.from_Tomcat01.duration_p95                      30              2   0.815305             1.148000  2021-03-07 07:44:00+08:00  1.408062
8     IG01           trace.self.duration_mean                      30              2  12.035910            16.852853  2021-03-07 07:33:00+08:00  1.400214
9     IG01     trace.to_Tomcat01.duration_p95                      30              2   0.690120             0.926800  2021-03-07 07:31:00+08:00  1.342955
10    IG01    trace.to_Tomcat02.duration_mean                      30              2   0.188276             0.252021  2021-03-07 07:42:00+08:00  1.338572
11    MG01      trace.from_Tomcat01.row_count                      30              2  47.550000            61.000000  2021-03-07 07:43:00+08:00  1.282860
12    MG01  trace.from_Tomcat02.duration_mean                      30              2   0.167448             0.213148  2021-03-07 07:42:00+08:00  1.272921
13    IG02     trace.to_Tomcat01.duration_p95                      30              2   1.042250             1.305300  2021-03-07 07:44:00+08:00  1.252387
14    IG01    trace.to_Tomcat03.duration_mean                      30              2   0.176748             0.217655  2021-03-07 07:43:00+08:00  1.231443
15    MG01  trace.from_dockerB2.duration_mean                      30              2   0.033209             0.040433  2021-03-07 07:43:00+08:00  1.217531
16    MG01   trace.from_Tomcat03.duration_p95                      30              2   0.917305             1.116300  2021-03-07 07:41:00+08:00  1.216934
17    IG02        trace.to_Tomcat02.row_count                      30              2  42.100000            51.000000  2021-03-07 07:43:00+08:00  1.211401
18    IG01     trace.to_Tomcat02.duration_p95                      30              2   1.028577             1.233800  2021-03-07 07:42:00+08:00  1.199521
19    IG01    trace.to_Tomcat01.duration_mean                      30              2   0.126800             0.151586  2021-03-07 07:31:00+08:00  1.195473,      timestamp cmdb_id                         trace_name     value       p95
0   1615073460    IG01    trace.to_Tomcat01.duration_mean  0.151586  0.126800
1   1615073460    IG01     trace.to_Tomcat01.duration_p95  0.926800  0.690120
2   1615073460    IG02    trace.to_Tomcat04.duration_mean  0.182522  0.161015
3   1615073460    IG02     trace.to_Tomcat04.duration_p95  1.118000  0.941360
4   1615073460    MG01  trace.from_Tomcat01.duration_mean  0.123500  0.121170
..         ...     ...                                ...       ...       ...
45  1615074240    MG01  trace.from_Tomcat01.duration_mean  0.185636  0.121170
46  1615074240    MG01   trace.from_Tomcat01.duration_p95  1.148000  0.815305
47  1615074300    MG01   trace.from_dockerA1.duration_p95  0.030700  0.019877
48  1615074300    MG01  trace.from_dockerA1.duration_mean  0.010750  0.009031
49  1615074360    IG02    trace.to_Tomcat04.duration_mean  0.168000  0.161015

[50 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Prepare and display the first 50 raw trace anomaly rows with a derived 'direction' column.
# Reuse existing kernel variable `trace_anomaly_rows`. Result will be stored in
# `trace_anomaly_rows_display` and displayed.

import numpy as np
import pandas as pd

# Safely access trace_anomaly_rows (may have been computed earlier)
try:
    ta = trace_anomaly_rows
except NameError:
    ta = pd.DataFrame(columns=['timestamp','_dt','cmdb_id','trace_name','value','p95'])

# Ensure _dt exists and sort by it ascending
if '_dt' not in ta.columns:
    # Attempt to create _dt from timestamp if possible
    if 'timestamp' in ta.columns and not ta['timestamp'].empty:
        ta = ta.copy()
        ta['_dt'] = pd.to_datetime(ta['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
    else:
        ta = ta.copy()
        ta['_dt'] = pd.NaT

ta_sorted = ta.sort_values('_dt').reset_index(drop=True).copy()

# Derive 'direction' based on trace_name substrings
# Priority: 'to' if contains 'to_', then 'from' if contains 'from_', then 'self' if contains 'self'
tn = ta_sorted.get('trace_name', pd.Series([], dtype=object))
cond_to = tn.str.contains('to_', na=False)
cond_from = tn.str.contains('from_', na=False)
cond_self = tn.str.contains('self', na=False)
ta_sorted['direction'] = np.select([cond_to, cond_from, cond_self],
                                   ['to', 'from', 'self'],
                                   default='other')

# Select requested columns and limit to 50 rows
cols = ['timestamp', '_dt', 'cmdb_id', 'trace_name', 'value', 'p95', 'direction']
# Ensure all columns exist
for c in cols:
    if c not in ta_sorted.columns:
        ta_sorted[c] = pd.NA

trace_anomaly_rows_display = ta_sorted[cols].head(50)

# Display the final compact table variable
trace_anomaly_rows_display

```
Out[4]:
```
# Summarize the raw trace anomaly rows (plain English) using kernel variables.
# This cell returns a single summary string `summary` as output.

import pandas as pd

# Access kernel variables safely
try:
    full = trace_anomaly_rows
except NameError:
    full = pd.DataFrame()

try:
    display_df = trace_anomaly_rows_display
except NameError:
    display_df = pd.DataFrame()

# Compute summary stats
if full.empty:
    summary = "No trace anomaly rows found in the specified window."
else:
    n_rows = int(full.shape[0])
    n_pairs = int(full[['cmdb_id','trace_name']].drop_duplicates().shape[0])
    earliest_ts = full['_dt'].min()
    earliest_str = earliest_ts.isoformat() if pd.notna(earliest_ts) else "None"
    # Top components by anomaly count
    comp_counts = full.groupby('cmdb_id').size().sort_values(ascending=False).head(5)
    comp_lines = [f"{ix} ({int(cnt)} rows)" for ix, cnt in comp_counts.items()]
    # Top trace names by anomaly count
    name_counts = full.groupby('trace_name').size().sort_values(ascending=False).head(5)
    name_lines = [f"{ix} ({int(cnt)} rows)" for ix, cnt in name_counts.items()]
    # Direction breakdown (derive from trace_name)
    tn = full['trace_name'].astype(str)
    dir_to = tn.str.contains('to_', na=False).sum()
    dir_from = tn.str.contains('from_', na=False).sum()
    dir_self = tn.str.contains('self', na=False).sum()
    dir_other = n_rows - (dir_to + dir_from + dir_self)
    # Build summary text
    lines = [
        "Trace anomaly rows summary for 2021-03-07 07:30:00+08 to 2021-03-07 08:00:00+08:",
        f"- Total anomaly rows: {n_rows}",
        f"- Distinct (cmdb_id, trace_name) pairs with anomalies: {n_pairs}",
        f"- Earliest anomaly timestamp: {earliest_str}",
        "",
        "- Top components by anomaly rows (top 5):",
    ]
    lines += [f"  - {s}" for s in comp_lines] if comp_lines else ["  - (none)"]
    lines += ["", "- Top trace names by anomaly rows (top 5):"]
    lines += [f"  - {s}" for s in name_lines] if name_lines else ["  - (none)"]
    lines += [
        "",
        "- Direction breakdown (based on trace_name):",
        f"  - to_: {int(dir_to)} rows",
        f"  - from_: {int(dir_from)} rows",
        f"  - self: {int(dir_self)} rows",
        f"  - other: {int(dir_other)} rows",
        "",
        f"The full filtered anomaly rows are stored in the kernel variable `trace_anomaly_rows`.",
        f"A compact displayed table (up to 50 rows) is in `trace_anomaly_rows_display` (columns: timestamp, _dt, cmdb_id, trace_name, value, p95, direction).",
    ]
    summary = "\n".join(lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                       _dt cmdb_id                         trace_name     value       p95 direction
0   1615073460 2021-03-07 07:31:00+08:00    IG01    trace.to_Tomcat01.duration_mean  0.151586  0.126800        to
1   1615073460 2021-03-07 07:31:00+08:00    IG01     trace.to_Tomcat01.duration_p95  0.926800  0.690120        to
2   1615073460 2021-03-07 07:31:00+08:00    IG02    trace.to_Tomcat04.duration_mean  0.182522  0.161015        to
3   1615073460 2021-03-07 07:31:00+08:00    IG02     trace.to_Tomcat04.duration_p95  1.118000  0.941360        to
4   1615073460 2021-03-07 07:31:00+08:00    MG01  trace.from_Tomcat01.duration_mean  0.123500  0.121170      from
..         ...                       ...     ...                                ...       ...       ...       ...
45  1615074240 2021-03-07 07:44:00+08:00    MG01  trace.from_Tomcat01.duration_mean  0.185636  0.121170      from
46  1615074240 2021-03-07 07:44:00+08:00    MG01   trace.from_Tomcat01.duration_p95  1.148000  0.815305      from
47  1615074300 2021-03-07 07:45:00+08:00    MG01   trace.from_dockerA1.duration_p95  0.030700  0.019877      from
48  1615074300 2021-03-07 07:45:00+08:00    MG01  trace.from_dockerA1.duration_mean  0.010750  0.009031      from
49  1615074360 2021-03-07 07:46:00+08:00    IG02    trace.to_Tomcat04.duration_mean  0.168000  0.161015        to

[50 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Filter trace_anomaly_rows for the earliest anomaly timestamp and compute 'implicated'.
# Reuse existing kernel variable `trace_anomaly_rows` (full filtered anomalies).
import pandas as pd
import numpy as np
import pytz
import re

# Safely access trace_anomaly_rows
try:
    ta = trace_anomaly_rows.copy()
except NameError:
    ta = pd.DataFrame(columns=['timestamp','_dt','cmdb_id','trace_name','value','p95'])

# Ensure _dt exists and is timezone-aware (UTC+8)
if '_dt' not in ta.columns:
    if 'timestamp' in ta.columns and not ta['timestamp'].empty:
        ta['_dt'] = pd.to_datetime(ta['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
    else:
        ta['_dt'] = pd.NaT

# If empty, create empty result with requested columns
if ta.empty:
    trace_earliest_rows = pd.DataFrame(columns=['timestamp','_dt','cmdb_id','trace_name','value','p95','direction','implicated'])
else:
    # Find earliest timestamp
    earliest_ts = ta['_dt'].min()

    # Filter rows at that earliest timestamp
    mask = ta['_dt'] == earliest_ts
    earliest_rows = ta.loc[mask].sort_values('_dt').reset_index(drop=True).copy()

    # Derive 'direction'
    tn = earliest_rows['trace_name'].astype(str)
    cond_to = tn.str.contains('to_', na=False)
    cond_from = tn.str.contains('from_', na=False)
    cond_self = tn.str.contains('self', na=False)
    earliest_rows['direction'] = np.select([cond_to, cond_from, cond_self],
                                           ['to', 'from', 'self'],
                                           default='other')

    # Derive 'implicated'
    # For 'to_' take substring after 'to_' up to next '.' or end
    # For 'from_' similarly. Else empty string.
    implicated = []
    for name in earliest_rows['trace_name'].astype(str):
        m_to = re.search(r'to_([^\.]+)', name)
        m_from = re.search(r'from_([^\.]+)', name)
        if m_to:
            implicated.append(m_to.group(1))
        elif m_from:
            implicated.append(m_from.group(1))
        else:
            implicated.append('')
    earliest_rows['implicated'] = implicated

    # Select requested columns and limit to 50 rows
    cols = ['timestamp','_dt','cmdb_id','trace_name','value','p95','direction','implicated']
    # Ensure all columns exist
    for c in cols:
        if c not in earliest_rows.columns:
            earliest_rows[c] = pd.NA

    trace_earliest_rows = earliest_rows[cols].head(50).copy()

# expose the result (compact, max 50 rows)
trace_earliest_rows

```
Out[5]:
```
# Summarize the earliest-anomaly rows (plain English) using the kernel variable `trace_earliest_rows`.
# The result will be a single summary string `summary` displayed as the output.
import pandas as pd

try:
    df = trace_earliest_rows.copy()
except NameError:
    df = pd.DataFrame()

if df.empty:
    summary = "No earliest-anomaly rows available."
else:
    # Earliest timestamp (all rows share the same _dt by construction)
    earliest_ts = df['_dt'].iloc[0]
    earliest_str = earliest_ts.isoformat() if pd.notna(earliest_ts) else str(earliest_ts)

    total_rows = int(df.shape[0])
    cmdbs = sorted(df['cmdb_id'].dropna().unique())
    directions = df['direction'].value_counts().to_dict()
    implicated_counts = df['implicated'].replace('', pd.NA).dropna().value_counts().to_dict()

    lines = [
        f"Earliest anomaly timestamp: {earliest_str}",
        f"Total anomaly rows at that timestamp: {total_rows}",
        "",
        f"Involved components (cmdb_id): {', '.join(cmdbs)}",
        "",
        "Direction breakdown:",
    ]
    for d, c in directions.items():
        lines.append(f"  - {d}: {c} row(s)")
    lines += [
        "",
        "Implicated target services (extracted from trace_name) and counts:",
    ]
    if implicated_counts:
        for target, cnt in implicated_counts.items():
            lines.append(f"  - {target}: {int(cnt)} row(s)")
    else:
        lines.append("  - (none extracted)")

    lines += [
        "",
        "Examples of anomalies at this timestamp (stored in kernel variable `trace_earliest_rows`):",
        "  - IG01 -> to Tomcat01 (duration_mean and duration_p95)",
        "  - IG02 -> to Tomcat04 (duration_mean and duration_p95)",
        "  - MG01 <- from Tomcat01, from dockerA1, from dockerB1",
        "",
        "Notes:",
        "  - `trace_earliest_rows` contains the displayed rows (timestamp, _dt, cmdb_id, trace_name, value, p95, direction, implicated).",
        "  - The full set of trace anomalies for the window is stored in `trace_anomaly_rows` for further analysis.",
    ]

    summary = "\n".join(lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                       _dt cmdb_id                         trace_name      value       p95 direction implicated
0  1615073460 2021-03-07 07:31:00+08:00    IG01    trace.to_Tomcat01.duration_mean   0.151586  0.126800        to   Tomcat01
1  1615073460 2021-03-07 07:31:00+08:00    IG01     trace.to_Tomcat01.duration_p95   0.926800  0.690120        to   Tomcat01
2  1615073460 2021-03-07 07:31:00+08:00    IG02    trace.to_Tomcat04.duration_mean   0.182522  0.161015        to   Tomcat04
3  1615073460 2021-03-07 07:31:00+08:00    IG02     trace.to_Tomcat04.duration_p95   1.118000  0.941360        to   Tomcat04
4  1615073460 2021-03-07 07:31:00+08:00    MG01  trace.from_Tomcat01.duration_mean   0.123500  0.121170      from   Tomcat01
5  1615073460 2021-03-07 07:31:00+08:00    MG01      trace.from_dockerA1.row_count  10.000000  9.550000      from   dockerA1
6  1615073460 2021-03-07 07:31:00+08:00    MG01  trace.from_dockerB1.duration_mean   0.145222  0.057638      from   dockerB1
7  1615073460 2021-03-07 07:31:00+08:00    MG01   trace.from_dockerB1.duration_p95   0.756400  0.172035      from   dockerB1```
```

In [ ]:
# In[6]:

# Focused metric anomaly analysis for components MG01, IG01, IG02 within the failure window.
# Reuse existing kernel variables where available: metric_df, metric_win, metric_counts_df
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')
tiny_eps = 1e-9

# Ensure metric_win exists (filtered to the window). If not, recreate from metric_df using UTC+8.
try:
    metric_win
except NameError:
    # create metric_win using previously loaded metric_df
    if 'metric_df' not in globals():
        raise RuntimeError("metric_df not found in kernel.")
    metric_win = metric_df.copy()
    if '_dt' not in metric_win.columns:
        metric_win['_dt'] = pd.to_datetime(metric_win['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure global p95 table exists: metric_counts_df (computed from full metric.csv earlier)
if 'metric_counts_df' not in globals():
    # Attempt to compute global p95 from full metric_df (must obey rule 9: compute global thresholds BEFORE filtering)
    # But ideally metric_counts_df is available; if not, compute now from metric_df
    if 'metric_df' not in globals():
        raise RuntimeError("metric_counts_df and metric_df are both unavailable in kernel.")
    gp = metric_df.groupby(['cmdb_id','kpi_name'])['value']
    metric_counts_df = gp.agg(count='count', p95=lambda s: float(np.nanpercentile(s.dropna(),95)) if s.dropna().shape[0] else np.nan).reset_index()

# Target components
targets = ['MG01','IG01','IG02']

# Filter metric_win to target components
metric_win_targets = metric_win[metric_win['cmdb_id'].isin(targets)].copy()

# Merge global p95 into windowed rows (inner join to keep only pairs with stored global p95)
merged = metric_win_targets.merge(
    metric_counts_df[['cmdb_id','kpi_name','p95']],
    on=['cmdb_id','kpi_name'],
    how='inner',
    validate='m:1'
).copy()

# If no merged rows, produce empty outputs
if merged.empty:
    metric_candidate_agg = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','anomaly_count',
        'earliest_anomaly_timestamp','p95','max_value_in_window','severity'
    ])
    metric_candidate_rows = pd.DataFrame(columns=['timestamp','_dt','cmdb_id','kpi_name','value','p95'])
else:
    # Determine anomaly per row: value > p95 (or >0 if p95==0)
    merged['is_anomaly'] = np.where(merged['p95'] == 0, merged['value'] > 0, merged['value'] > merged['p95'])

    # Aggregations per (cmdb_id, kpi_name)
    grp = merged.groupby(['cmdb_id','kpi_name'], dropna=False)
    total_points = grp['value'].count().rename('total_points_in_window')
    anomaly_counts = grp['is_anomaly'].sum().rename('anomaly_count')
    p95_series = grp['p95'].first().rename('p95')
    max_values = grp['value'].max().rename('max_value_in_window')

    agg = pd.concat([total_points, anomaly_counts, p95_series, max_values], axis=1).reset_index()

    # earliest anomaly timestamp per group (from merged rows that are anomalies)
    anomalies_only = merged[merged['is_anomaly']].copy()
    if not anomalies_only.empty:
        earliest_anom = anomalies_only.groupby(['cmdb_id','kpi_name'])['_dt'].min().rename('earliest_anomaly_timestamp').reset_index()
        agg = agg.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')
    else:
        agg['earliest_anomaly_timestamp'] = pd.NaT

    # Keep only rows with anomaly_count > 0
    agg = agg[agg['anomaly_count'] > 0].copy()

    # Compute severity
    agg['severity'] = agg['max_value_in_window'] / agg['p95'].replace(0, tiny_eps)

    # Round numeric columns for compactness
    agg['p95'] = agg['p95'].astype(float).round(6)
    agg['max_value_in_window'] = agg['max_value_in_window'].astype(float).round(6)
    agg['severity'] = agg['severity'].astype(float).round(6)

    # Order and limit to top 20
    metric_candidate_agg = agg.sort_values(['anomaly_count','severity'], ascending=[False, False]).head(20).reset_index(drop=True)

    # Select top 5 pairs for raw rows
    top5 = metric_candidate_agg[['cmdb_id','kpi_name']].head(5).copy()

    # For each top pair, collect anomaly rows within the window, sorted by timestamp ascending, limit 50 per KPI
    rows_list = []
    for _, r in top5.iterrows():
        c = r['cmdb_id']
        k = r['kpi_name']
        mask = (merged['cmdb_id'] == c) & (merged['kpi_name'] == k) & (merged['is_anomaly'])
        subset = merged.loc[mask].sort_values('_dt').head(50)
        if not subset.empty:
            rows_list.append(subset[['timestamp','_dt','cmdb_id','kpi_name','value','p95']])

    if rows_list:
        metric_candidate_rows = pd.concat(rows_list, ignore_index=True)
    else:
        metric_candidate_rows = pd.DataFrame(columns=['timestamp','_dt','cmdb_id','kpi_name','value','p95'])

# Final displayed outputs (compact): aggregated table and concatenated raw anomaly rows
metric_candidate_agg, metric_candidate_rows

```
Out[6]:
```
# Summarize metric anomalies for MG01, IG01, IG02 within the window using kernel results.
# This cell produces a single plain-English summary string `summary` as output.
import pandas as pd

# Safely access kernel variables
metric_agg = globals().get('metric_candidate_agg', pd.DataFrame())
metric_rows = globals().get('metric_candidate_rows', pd.DataFrame())

if metric_agg.empty:
    summary = "No metric anomalies found for MG01, IG01, or IG02 in the specified window."
else:
    n_pairs = int(metric_agg.shape[0])
    n_rows = int(metric_rows.shape[0])
    # Build top list (up to 5)
    top_lines = []
    for _, r in metric_agg.head(5).iterrows():
        comp = r['cmdb_id']
        kpi = r['kpi_name']
        cnt = int(r['anomaly_count'])
        p95 = float(r['p95'])
        maxv = float(r['max_value_in_window'])
        sev = float(r['severity'])
        ts = r.get('earliest_anomaly_timestamp', pd.NaT)
        ts_str = ts.isoformat() if pd.notna(ts) else "None"
        top_lines.append(f"{comp} | {kpi} | anomalies={cnt} | p95={p95:.6g} | max={maxv:.6g} | sev={sev:.6g} | first={ts_str}")

    # Components involved
    comps = sorted(metric_agg['cmdb_id'].unique().tolist())
    summary_lines = [
        "Metric anomaly summary for components MG01, IG01, IG02 (window 2021-03-07 07:30 to 08:00 +08):",
        f"- (cmdb_id,kpi) pairs with anomalies: {n_pairs}",
        f"- Total raw anomaly rows returned for top pairs: {n_rows}",
        f"- Components with anomalies among the three: {', '.join(comps)}",
        "",
        "Top (component | kpi_name | anomaly_count | p95 | max_value | severity | earliest):"
    ]
    summary_lines += [f"  - {ln}" for ln in top_lines]
    summary_lines += [
        "",
        "Observations:",
        " - All detected anomalies for these three components are on IG01 (several CPU metrics and JVM heap used).",
        " - Highest severity in this set: IG01 OSLinux-CPU_CPU_CPUWio (severity ~2.17) and IG01 JVM heap used (severity ~1.08).",
        " - The raw anomaly rows for the selected top pairs are stored in `metric_candidate_rows` for follow-up."
    ]
    summary = "\n".join(summary_lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                                   kpi_name  total_points_in_window  anomaly_count           p95  max_value_in_window earliest_anomaly_timestamp  severity
0    IG01                     OSLinux-CPU_CPU_CPUWio                      31              2  2.640000e-01         5.737810e-01  2021-03-07 07:40:00+08:00  2.173413
1    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed                      31              2  7.802229e+08         8.423134e+08  2021-03-07 07:50:00+08:00  1.079580
2    IG01                 OSLinux-CPU_CPU_CPUSysTime                      31              2  3.928820e-01         4.115000e-01  2021-03-07 07:36:00+08:00  1.047388
3    IG01                    OSLinux-CPU_CPU_CPULoad                      31              2  1.025000e+00         1.070000e+00  2021-03-07 07:34:00+08:00  1.043902
4    IG01                OSLinux-CPU_CPU_CPUUserTime                      31              2  2.518240e+01         2.521515e+01  2021-03-07 07:43:00+08:00  1.001301
5    IG01                OSLinux-CPU_CPU_CPUidleutil                      31              2  7.456300e+01         7.463830e+01  2021-03-07 07:32:00+08:00  1.001010,     timestamp                       _dt cmdb_id                                   kpi_name         value           p95
0  1615074000 2021-03-07 07:40:00+08:00    IG01                     OSLinux-CPU_CPU_CPUWio  2.952000e-01  2.640000e-01
1  1615074060 2021-03-07 07:41:00+08:00    IG01                     OSLinux-CPU_CPU_CPUWio  5.737810e-01  2.640000e-01
2  1615074600 2021-03-07 07:50:00+08:00    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.030927e+08  7.802229e+08
3  1615074660 2021-03-07 07:51:00+08:00    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.423134e+08  7.802229e+08
4  1615073760 2021-03-07 07:36:00+08:00    IG01                 OSLinux-CPU_CPU_CPUSysTime  4.115000e-01  3.928820e-01
5  1615074780 2021-03-07 07:53:00+08:00    IG01                 OSLinux-CPU_CPU_CPUSysTime  4.074340e-01  3.928820e-01
6  1615073640 2021-03-07 07:34:00+08:00    IG01                    OSLinux-CPU_CPU_CPULoad  1.070000e+00  1.025000e+00
7  1615074780 2021-03-07 07:53:00+08:00    IG01                    OSLinux-CPU_CPU_CPULoad  1.030000e+00  1.025000e+00
8  1615074180 2021-03-07 07:43:00+08:00    IG01                OSLinux-CPU_CPU_CPUUserTime  2.519130e+01  2.518240e+01
9  1615074780 2021-03-07 07:53:00+08:00    IG01                OSLinux-CPU_CPU_CPUUserTime  2.521515e+01  2.518240e+01)```
```

In [ ]:
# In[7]:

# Aggregate trace anomalies by implicated target extracted from trace_anomaly_rows.
# Store full aggregated DataFrame and a compact (top 20) table in kernel variables.
import pandas as pd
import numpy as np
import re
import pytz

tiny_eps = 1e-9
tz = pytz.timezone('Asia/Shanghai')

# Access trace_anomaly_rows from kernel
try:
    taf = trace_anomaly_rows.copy()
except NameError:
    taf = pd.DataFrame(columns=['timestamp','_dt','cmdb_id','trace_name','value','p95'])

# Ensure _dt exists and is timezone-aware (UTC+8)
if '_dt' not in taf.columns:
    if 'timestamp' in taf.columns and not taf['timestamp'].empty:
        taf['_dt'] = pd.to_datetime(taf['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        taf['_dt'] = pd.NaT

# Derive 'implicated' (substring after 'to_' or 'from_' up to next '.' or end)
def extract_implicated(name):
    if not isinstance(name, str):
        return ''
    m = re.search(r'to_([^\.]+)', name)
    if m:
        return m.group(1)
    m = re.search(r'from_([^\.]+)', name)
    if m:
        return m.group(1)
    return ''

taf['implicated'] = taf['trace_name'].astype(str).apply(extract_implicated)

# Compute per-row severity = value / max(p95, tiny_eps)
taf['p95_safe'] = taf['p95'].replace(0, tiny_eps).astype(float)
taf['severity'] = taf['value'].astype(float) / taf['p95_safe']

# Filter to rows that have a non-empty implicated target
taf_imp = taf[taf['implicated'].str.len() > 0].copy()

# Aggregation by implicated
if taf_imp.empty:
    trace_implicated_agg_full = pd.DataFrame(columns=[
        'implicated','anomaly_row_count','earliest_anomaly_timestamp',
        'latest_anomaly_timestamp','distinct_cmdb_ids_list','max_severity'
    ])
else:
    grp = taf_imp.groupby('implicated', dropna=False)
    agg = grp.agg(
        anomaly_row_count=('implicated','size'),
        earliest_anomaly_timestamp=('_dt','min'),
        latest_anomaly_timestamp=('_dt','max'),
        distinct_cmdb_ids_list=('cmdb_id', lambda s: sorted(s.dropna().unique().tolist())),
        max_severity=('severity','max')
    ).reset_index()

    # Round max_severity for compactness
    agg['max_severity'] = agg['max_severity'].astype(float).round(6)

    # Sort by anomaly_row_count desc then max_severity desc
    trace_implicated_agg_full = agg.sort_values(['anomaly_row_count','max_severity'], ascending=[False, False]).reset_index(drop=True)

# Prepare compact top-20 table to display and store
trace_implicated_agg = trace_implicated_agg_full.head(20).copy()

# Keep both DataFrames in kernel for later steps:
# - trace_implicated_agg (compact top-20)
# - trace_implicated_agg_full (full aggregation)

# Display compact result and the full aggregation size for reference
trace_implicated_agg, trace_implicated_agg_full.shape

```
Out[7]:
```
```python
# Produce a plain-English summary string of the implicated-target aggregation results.
# Reuse existing kernel variable `trace_implicated_agg` (compact top-20) and `trace_implicated_agg_full` (full).
try:
    agg = trace_implicated_agg.copy()
except NameError:
    agg = None

if agg is None or agg.empty:
    summary = "No implicated targets found in trace anomalies for the window."
else:
    lines = []
    lines.append("Aggregated trace-anomaly summary by implicated target (compact top results):")
    lines.append("")
    for _, r in agg.iterrows():
        imp = r['implicated']
        cnt = int(r['anomaly_row_count'])
        earliest = r['earliest_anomaly_timestamp']
        latest = r['latest_anomaly_timestamp']
        cms = r['distinct_cmdb_ids_list']
        maxsev = float(r['max_severity'])
        lines.append(f"- {imp}: {cnt} anomaly rows; earliest={earliest}; latest={latest}; reported_by={cms}; max_severity={maxsev:.6g}")
    lines.append("")
    lines.append("Key observations:")
    lines.append("- The top implicated targets are Tomcat services (Tomcat02, Tomcat04, Tomcat01, Tomcat03) with 16–18 anomaly rows each,")
    lines.append("  all reported by multiple front-end components [IG01, IG02, MG01].")
    lines.append("- Some backend containers (dockerB1, dockerB2, dockerA1) are implicated as well, primarily reported by MG01.")
    lines.append("- The highest per-target severity values are for dockerB1 (~4.40) and dockerB2 (~3.52), indicating the largest relative exceedances of their global P95s.")
    lines.append("")
    lines.append("Data availability:")
    lines.append("- The compact top-20 implicated table is stored in `trace_implicated_agg`.")
    lines.append("- The full aggregation (all implicated targets) is stored in `trace_implicated_agg_full` for further analysis.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  implicated  anomaly_row_count earliest_anomaly_timestamp  latest_anomaly_timestamp distinct_cmdb_ids_list  max_severity
0   Tomcat02                 18  2021-03-07 07:42:00+08:00 2021-03-07 07:59:00+08:00     [IG01, IG02, MG01]      1.338572
1   Tomcat04                 17  2021-03-07 07:31:00+08:00 2021-03-07 07:54:00+08:00     [IG01, IG02, MG01]      1.594783
2   Tomcat01                 17  2021-03-07 07:31:00+08:00 2021-03-07 07:57:00+08:00     [IG01, IG02, MG01]      1.532029
3   Tomcat03                 16  2021-03-07 07:41:00+08:00 2021-03-07 07:58:00+08:00     [IG01, IG02, MG01]      1.231443
4   dockerB1                  6  2021-03-07 07:31:00+08:00 2021-03-07 07:58:00+08:00                 [MG01]      4.396780
5   dockerA1                  6  2021-03-07 07:31:00+08:00 2021-03-07 07:59:00+08:00                 [MG01]      1.544499
6   dockerB2                  4  2021-03-07 07:33:00+08:00 2021-03-07 07:51:00+08:00                 [MG01]      3.522379, (7, 6))```
```

In [ ]:
# In[8]:

# Aggregate and extract trace anomaly rows for implicated targets dockerB1 and dockerB2.
# Reuse existing kernel variable `trace_anomaly_rows`. Store outputs in:
# - trace_docker_agg (aggregated per implicated)
# - trace_docker_rows (full filtered anomaly rows)
# Also produce a compact display variable trace_docker_rows_display (first 50 rows).

import pandas as pd
import numpy as np
import pytz
import re

tiny_eps = 1e-9
tz = pytz.timezone('Asia/Shanghai')

# Safely access trace_anomaly_rows
try:
    ta = trace_anomaly_rows.copy()
except NameError:
    ta = pd.DataFrame(columns=['timestamp','_dt','cmdb_id','trace_name','value','p95'])

# Ensure _dt exists and is timezone-aware (UTC+8)
if '_dt' not in ta.columns:
    if 'timestamp' in ta.columns and not ta['timestamp'].empty:
        ta['_dt'] = pd.to_datetime(ta['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        ta['_dt'] = pd.NaT

# Ensure p95 and value exist
if 'p95' not in ta.columns:
    ta['p95'] = np.nan
if 'value' not in ta.columns:
    ta['value'] = np.nan

# Derive 'implicated' if not present
if 'implicated' not in ta.columns:
    def extract_implicated(name):
        if not isinstance(name, str):
            return ''
        m = re.search(r'to_([^\.]+)', name)
        if m:
            return m.group(1)
        m = re.search(r'from_([^\.]+)', name)
        if m:
            return m.group(1)
        return ''
    ta['implicated'] = ta['trace_name'].astype(str).apply(extract_implicated)

# Compute per-row severity = value / max(p95, tiny_eps)
ta['p95_safe'] = ta['p95'].replace(0, tiny_eps).astype(float)
ta['severity'] = ta['value'].astype(float) / ta['p95_safe']

# Filter for dockerB1 and dockerB2
targets = ['dockerB1', 'dockerB2']
mask = ta['implicated'].isin(targets)
trace_docker_rows = ta[mask].copy()  # full filtered rows stored in kernel

# Aggregation by implicated target
if trace_docker_rows.empty:
    trace_docker_agg = pd.DataFrame(columns=[
        'implicated', 'anomaly_row_count', 'earliest_anomaly_timestamp',
        'latest_anomaly_timestamp', 'distinct_cmdb_ids_list', 'max_severity'
    ])
else:
    grp = trace_docker_rows.groupby('implicated', dropna=False)
    agg = grp.agg(
        anomaly_row_count=('implicated','size'),
        earliest_anomaly_timestamp=('_dt','min'),
        latest_anomaly_timestamp=('_dt','max'),
        distinct_cmdb_ids_list=('cmdb_id', lambda s: sorted(s.dropna().unique().tolist())),
        max_severity=('severity','max')
    ).reset_index()
    agg['max_severity'] = agg['max_severity'].astype(float).round(6)
    trace_docker_agg = agg.sort_values(['anomaly_row_count','max_severity'], ascending=[False, False]).reset_index(drop=True)

# Prepare compact raw display limited to 50 rows, sorted by timestamp ascending
cols_display = ['timestamp', '_dt', 'cmdb_id', 'trace_name', 'value', 'p95', 'implicated']
for c in cols_display:
    if c not in trace_docker_rows.columns:
        trace_docker_rows[c] = pd.NA
trace_docker_rows_display = trace_docker_rows.sort_values('_dt')[cols_display].head(50)

# Expose the compact outputs (these variables will be the cell output)
trace_docker_agg, trace_docker_rows_display

```
Out[8]:
```
Summary of trace anomalies for implicated targets dockerB1 and dockerB2 (window 2021-03-07 07:30–08:00 +08):

- Aggregated results:
  - dockerB1: 6 anomaly rows, earliest = 2021-03-07 07:31:00+08, latest = 2021-03-07 07:58:00+08, reported by [MG01], max_severity ≈ 4.39678.
  - dockerB2: 4 anomaly rows, earliest = 2021-03-07 07:33:00+08, latest = 2021-03-07 07:51:00+08, reported by [MG01], max_severity ≈ 3.52238.

- Raw anomaly observations (from trace_docker_rows, sorted ascending):
  - Earliest dockerB1 anomalies occur at 07:31:00 (MG01: trace.from_dockerB1.duration_mean and trace.from_dockerB1.duration_p95).
  - Earliest dockerB2 anomaly occurs at 07:33:00 (MG01: trace.from_dockerB2.duration_p95).
  - Subsequent dockerB1 anomalies appear at ~07:46, ~07:53, ~07:54, and ~07:58.
  - Subsequent dockerB2 anomalies appear at ~07:43 (two rows) and ~07:51.

- Interpretation for next steps:
  - The absolute first anomalous point among these targets is dockerB1 at 2021-03-07 07:31:00+08.
  - All anomalies for dockerB1 and dockerB2 were reported by MG01 in this window.
  - Anomalies are not strictly a single consecutive block; they occur at multiple timestamps across the window (some clustered, some spaced).
  - Stored variables for follow-up:
    - Aggregated table: trace_docker_agg
    - Full filtered raw rows: trace_docker_rows

If you want, I can (a) show the full trace_docker_rows table, (b) check whether any anomalies are strictly consecutive minute-by-minute for each target, or (c) extract the exact sequence of timestamps per target. Which do you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(  implicated  anomaly_row_count earliest_anomaly_timestamp  latest_anomaly_timestamp distinct_cmdb_ids_list  max_severity
0   dockerB1                  6  2021-03-07 07:31:00+08:00 2021-03-07 07:58:00+08:00                 [MG01]      4.396780
1   dockerB2                  4  2021-03-07 07:33:00+08:00 2021-03-07 07:51:00+08:00                 [MG01]      3.522379,        timestamp                       _dt cmdb_id                         trace_name      value        p95 implicated
95    1615073460 2021-03-07 07:31:00+08:00    MG01  trace.from_dockerB1.duration_mean   0.145222   0.057638   dockerB1
96    1615073460 2021-03-07 07:31:00+08:00    MG01   trace.from_dockerB1.duration_p95   0.756400   0.172035   dockerB1
199   1615073580 2021-03-07 07:33:00+08:00    MG01   trace.from_dockerB2.duration_p95   0.128750   0.036552   dockerB2
698   1615074180 2021-03-07 07:43:00+08:00    MG01  trace.from_dockerB2.duration_mean   0.035075   0.033209   dockerB2
699   1615074180 2021-03-07 07:43:00+08:00    MG01   trace.from_dockerB2.duration_p95   0.054800   0.036552   dockerB2
845   1615074360 2021-03-07 07:46:00+08:00    MG01  trace.from_dockerB1.duration_mean   0.058350   0.057638   dockerB1
1098  1615074660 2021-03-07 07:51:00+08:00    MG01  trace.from_dockerB2.duration_mean   0.040433   0.033209   dockerB2
1197  1615074780 2021-03-07 07:53:00+08:00    MG01      trace.from_dockerB1.row_count  62.000000  59.300000   dockerB1
1247  1615074840 2021-03-07 07:54:00+08:00    MG01      trace.from_dockerB1.row_count  65.000000  59.300000   dockerB1
1446  1615075080 2021-03-07 07:58:00+08:00    MG01   trace.from_dockerB1.duration_p95   0.225000   0.172035   dockerB1)```
```

In [ ]:
# In[9]:

# Analyze MG01 metrics within the failure window using precomputed global P95 thresholds.
# Reuse kernel variables where available: metric_win, metric_df, metric_counts_df
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')
tiny_eps = 1e-9

# Ensure metric_win exists (window-filtered metrics). If not, derive from metric_df.
if 'metric_win' not in globals():
    if 'metric_df' not in globals():
        raise RuntimeError("metric_df not found in kernel.")
    metric_win = metric_df.copy()
    if '_dt' not in metric_win.columns:
        metric_win['_dt'] = pd.to_datetime(metric_win['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure global p95 table exists (computed across entire metric series before window filtering)
if 'metric_counts_df' not in globals():
    if 'metric_df' not in globals():
        raise RuntimeError("metric_counts_df and metric_df are both unavailable in kernel.")
    gp = metric_df.groupby(['cmdb_id','kpi_name'])['value']
    metric_counts_df = gp.agg(count='count', p95=lambda s: float(np.nanpercentile(s.dropna(),95)) if s.dropna().shape[0] else np.nan).reset_index()

# Focus on MG01
mg = 'MG01'
metric_win_mg = metric_win[metric_win['cmdb_id'] == mg].copy()

# Merge global p95 into windowed MG01 rows (inner join to keep only pairs with a stored global p95)
merged_mg = metric_win_mg.merge(
    metric_counts_df[['cmdb_id','kpi_name','p95']],
    on=['cmdb_id','kpi_name'],
    how='inner',
    validate='m:1'
).copy()

# If no rows, create empty outputs
if merged_mg.empty:
    mg01_metric_agg = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','anomaly_count',
        'earliest_anomaly_timestamp','p95','max_value_in_window','severity'
    ])
    mg01_metric_rows = pd.DataFrame(columns=['timestamp','_dt','cmdb_id','kpi_name','value','p95'])
    mg01_metric_rows_display = mg01_metric_rows.head(50)
else:
    # Determine anomalies: value > p95 (or >0 if p95 == 0)
    merged_mg['is_anomaly'] = np.where(merged_mg['p95'] == 0, merged_mg['value'] > 0, merged_mg['value'] > merged_mg['p95'])

    # Aggregations per KPI
    grp = merged_mg.groupby(['cmdb_id','kpi_name'], dropna=False)
    total_points = grp['value'].count().rename('total_points_in_window')
    anomaly_counts = grp['is_anomaly'].sum().rename('anomaly_count')
    p95_series = grp['p95'].first().rename('p95')
    max_values = grp['value'].max().rename('max_value_in_window')

    agg = pd.concat([total_points, anomaly_counts, p95_series, max_values], axis=1).reset_index()

    # Earliest anomaly timestamp per KPI (from anomaly rows)
    anomalies_only = merged_mg[merged_mg['is_anomaly']].copy()
    if not anomalies_only.empty:
        earliest_anom = anomalies_only.groupby(['cmdb_id','kpi_name'])['_dt'].min().rename('earliest_anomaly_timestamp').reset_index()
        agg = agg.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')
    else:
        agg['earliest_anomaly_timestamp'] = pd.NaT

    # Keep only KPI rows with anomaly_count > 0
    agg = agg[agg['anomaly_count'] > 0].copy()

    # Compute severity
    agg['severity'] = agg['max_value_in_window'] / agg['p95'].replace(0, tiny_eps)

    # Round for compactness
    agg['p95'] = agg['p95'].astype(float).round(6)
    agg['max_value_in_window'] = agg['max_value_in_window'].astype(float).round(6)
    agg['severity'] = agg['severity'].astype(float).round(6)

    # Order and limit to top 20
    mg01_metric_agg = agg.sort_values(['anomaly_count','severity'], ascending=[False, False]).head(20).reset_index(drop=True)

    # Select top 3 KPIs (by anomaly_count then severity)
    top3 = mg01_metric_agg[['cmdb_id','kpi_name']].head(3)

    # Collect raw anomaly rows for these top3 KPIs (within the window)
    rows_list = []
    for _, r in top3.iterrows():
        c = r['cmdb_id']
        k = r['kpi_name']
        mask = (merged_mg['cmdb_id'] == c) & (merged_mg['kpi_name'] == k) & (merged_mg['is_anomaly'])
        subset = merged_mg.loc[mask].sort_values('_dt')
        if not subset.empty:
            # select requested columns
            rows_list.append(subset[['timestamp','_dt','cmdb_id','kpi_name','value','p95']])

    if rows_list:
        mg01_metric_rows = pd.concat(rows_list, ignore_index=True)
    else:
        mg01_metric_rows = pd.DataFrame(columns=['timestamp','_dt','cmdb_id','kpi_name','value','p95'])

    # Prepare display-limited version (first 50 rows)
    mg01_metric_rows_display = mg01_metric_rows.sort_values('_dt').head(50)

# Expose compact outputs (stored in kernel variables as required)
mg01_metric_agg, mg01_metric_rows_display

```
Out[9]:
```
```python
# Plain-English summary of MG01 metric analysis results (produced earlier and stored in kernel variables).
try:
    empty_agg = mg01_metric_agg.empty
    empty_rows = mg01_metric_rows.empty
except NameError:
    empty_agg = True
    empty_rows = True

if empty_agg and empty_rows:
    summary = (
        "Result: No metric-level anomalies detected for MG01 within 2021-03-07 07:30:00+08 to 2021-03-07 08:00:00+08. "
        "Both mg01_metric_agg and mg01_metric_rows are empty DataFrames. "
        "Therefore MG01 does NOT show metric KPI exceedances (value > global P95) in that window. "
        "Note: trace anomalies referencing dockerB1/dockerB2 were reported by MG01, but MG01's own metrics show no anomalies."
    )
else:
    summary = (
        "Result: MG01 metric anomalies were found in the window. "
        "Aggregated anomalies are stored in `mg01_metric_agg` and raw anomaly rows in `mg01_metric_rows`."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, total_points_in_window, anomaly_count, earliest_anomaly_timestamp, p95, max_value_in_window, severity]
Index: [], Empty DataFrame
Columns: [timestamp, _dt, cmdb_id, kpi_name, value, p95]
Index: [])```
```